# HyperStreamDB: Full Pipeline Presentation

**Version:** 0.1.9 (Alpha)  
**Project:** HyperStreamDB - Serverless Index-Streaming Data Lake

This masterclass takes you from zero to a production-grade RAG pipeline, demonstrating hardware acceleration, hybrid search, and serverless table management.

## 1. Setup & Environment

We initialize our environment and ensure all vector search dependencies are loaded.

In [1]:
import hyperstreamdb as hdb
import pyarrow as pa
import pandas as pd
import numpy as np
import os, shutil, requests, time, warnings
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 150)

print("Environment ready.")

Environment ready.


## 2. Hardware Acceleration

HyperStreamDB can utilize Intel GPUs for massively parallel index construction. We check for available hardware here.

GPU support for Nvidia, Intel, Apple MPS, AMD(ROCm - untested)

In [2]:
import torch
device = "cpu"
if torch.cuda.is_available():
    device = "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")
ctx = hdb.Device(device)


Using device: mps


## 3. Data Preparation (AG News)

We load the AG News dataset and generate semantic embeddings using a 384-dimensional SBERT model.

In [3]:
print("Loading dataset...")
dataset = load_dataset("ag_news", split="test[:500]")
df = pd.DataFrame(dataset)
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_map)

print("Generating embeddings...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]
print(f"Prepared {len(df)} records with {len(df['embedding'][0])}D vectors.")

df["id"] = range(len(df))


Loading dataset...
Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prepared 500 records with 384D vectors.


## 4. Table Creation (Serverless Iceberg)

HyperStreamDB tables are serverless and use standard Iceberg/Parquet formats with sidecar indices.

In [4]:
TABLE_URI = "file:///tmp/hdb_masterclass"
if os.path.exists("/tmp/hdb_masterclass"): shutil.rmtree("/tmp/hdb_masterclass")

schema = pa.schema([
    ('id', pa.int32()), 
    ('category', pa.string()),
    ('text', pa.string()),
    ('embedding', pa.list_(pa.float32(), 384))
])

table = hdb.Table.create(TABLE_URI, schema, device=ctx)
print(f"Table created at {TABLE_URI}")

Table created at file:///tmp/hdb_masterclassCommitted Manifest v1 (Schema Update)



DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)


## 5. Ingestion & Indexing

We write the data and building HNSW vector indices and Inverted scalar indices simultaneously.

In [5]:
table.add_index_columns(["embedding", "category", "text"])
table.write(df[["id", "category", "text", "embedding"]])
table.commit()
print("Data ingested and indexed successfully.")

Optimizing data layout: Shuffling rows by vector similarity (LanceDB-style)...
Written data to /tmp/hdb_masterclass/seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.parquet (500 rows)
Building indexes for batch of 500 rows
Indexing Int32 column: id
Indexing String column: text
Indexing String column: category
  Found 4 unique values
Indexing Vector column: embedding (type=FixedSizeList(Field { data_type: Float32, nullable: true }, 384))
  Found 500 unique values
String Inverted Index written to /tmp/hdb_masterclass/seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.category.inv.parquet
Building HNSW-IVF index (blocking): 500 vectors, 384 dims, use_pq=false
Building HNSW-IVF index: 500 vectors, 4 clusters, 384 dims, M=16, use_pq=false
String Inverted Index written to /tmp/hdb_masterclass/seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.text.inv.parquet
Committed Manifest: _manifest/v2.json
  - K-Means took: 4.09ms (5 iterations, 4 clusters)
  - Grouping vectors took: 324.38µs
  - Building HNSW graphs took: 7.93m

DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: to_manifest_entry: segment_id=seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab, generated index_files=[IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.category.inv.parquet", index_type: "inverted", column_name: Some("category"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_

## 6. Hybrid Search Performance

We perform a semantic search combined with a scalar filter to find 'Space' news in the 'Sci/Tech' category.

In [6]:
query_vec = list(model.encode("Latest milestones in space exploration"))

results = (table.filter("category = 'Sci/Tech'")
                 .vector_search(query_vec, k=3)
                 .to_pandas())

print("Hybrid Search Results (Sci/Tech + Vector):")
display(results[["category", "text", "distance"]])

Hybrid Search Results (Sci/Tech + Vector):


DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)
DEBUG: Total entries to prune: 1
DEBUG: Entry seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.parquet: partitions={}, index_files=[IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.category.inv.parquet", index_type: "inverted", column_name: Some("category"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.id.inv.parquet", index_type: "inverted", column_name: Some("id"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.text.inv.parquet", index_type: "inverted", column_name: Some("text"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.embedding.cluster_2.hnsw.graph", index_type: "vector", column_name: Some("embedding"), blob

,category,text,distance
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.113737
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.254462
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.255060


## 7. SQL & pgvector Interface

HyperStreamDB supports standard SQL with pgvector-compatible operators for vector distance.

In [7]:
session = hdb.Session()
session.register("news", table)

query_vec_str = str(query_vec).replace(' ', '')
sql_df = session.sql(f"""
    SELECT category, text, 
           embedding <-> '{query_vec_str}'::vector AS dist
    FROM news
    WHERE category = 'Sci/Tech'
    ORDER BY dist ASC LIMIT 3
""")
display(sql_df.to_pandas())

DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)
DEBUG: Searching for index for column: category. Available index files: [IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.category.inv.parquet", index_type: "inverted", column_name: Some("category"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.id.inv.parquet", index_type: "inverted", column_name: Some("id"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.text.inv.parquet", index_type: "inverted", column_name: Some("text"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_dd564b4a-

,category,text,dist
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.055338
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.120027
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.120294


## 8. Vector Aggregations

Calculate semantic centroids for entire categories directly in SQL.

In [8]:
agg_df = session.sql("""
    SELECT category, 
           vector_avg(embedding) AS centroid,
           COUNT(*) as count
    FROM news
    GROUP BY category
    order by count asc
""").to_pandas()
display(agg_df[["category", "count"]])

DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)


,category,count
0,Business,106
1,World,122
2,Sci/Tech,127
3,Sports,145


## 9. Table Maintenance

Keep the data lake healthy by compacting small files and vacuuming history.

In [9]:
print(f"Files before: {len(table.manifest().entries)}")
table.compact()
table.commit()
print(f"Files after: {len(table.manifest().entries)}")

table.vacuum(retention_versions=1)
print("Vacuum complete.")

Files before: 0
Starting compaction with strategy: binpack
Files after: 0
Found candidate: seg_dd564b4a-531b-40fa-8ec0-b804d85c90ab.parquet (1194034 bytes)
Plan: Identified 1 bins to compact.
Executing with parallelism: 4
Written data to /var/folders/96/k023nzcn1nzc1dklrz5_k1t80000gn/T/hyperstream_compact_99d89eb6-40e9-4c87-b12b-40c46cbfb9b8/compacted_20260404210904_99d89eb6-40e9-4c87-b12b-40c46cbfb9b8.parquet (500 rows)
Compacted bin 1 -> compacted_20260404210904_99d89eb6-40e9-4c87-b12b-40c46cbfb9b8.parquet
Committing Batch: +1 entries, -1 paths
Committed Manifest: _manifest/v4.json
Vacuum complete.


DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 3 on disk at Path { raw: "_manifest/v3.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v3 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 4 on disk at Path { raw: "_manifest/v4.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v4 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 4 on disk at Path { raw: "_manifest/v4.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v4 (entries=0)


## 10. Advanced Partitioning

Scale to massive datasets using identity partitioning on the category field.

In [10]:
PART_URI = "file:///tmp/part_demo"
if os.path.exists("/tmp/part_demo"): shutil.rmtree("/tmp/part_demo")

partition_spec = {
    "fields": [{"name": "category", "transform": "identity", "source_id": 1, "field_id": 1000}]
}

part_table = hdb.Table.create_partitioned(PART_URI, schema, partition_spec)
part_table.write(df[:100])
part_table.commit()

print("Partitioned physical layout:")
for entry in part_table.manifest().entries:
    print(f" - {entry.file_path.split('/')[-2:]}")

DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: to_manifest_entry: segment_id=seg_6efbc46e-fa4d-4fef-8db0-d81a2c3f8617, generated index_files=[IndexFile { file_path: "seg_6efbc46e-fa4d-4fef-8db0-d81a2c3f8617.text.inv.parquet", index_type: "inverted", column_name: Some("text"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_6efbc46e-fa4d-4fef-8db0-d81a2c3f8617.id.inv.parquet", index_type: "inverted", column_name: Some("id"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_

DEBUG: Rust Table created with runtime: true
Optimizing data layout: Shuffling rows by vector similarity (LanceDB-style)...
Written data to /tmp/part_demo/seg_0b9d4370-292e-4f0d-b7f1-21832b0de151.parquet (1 rows)
Written data to /tmp/part_demo/seg_b3baed84-6ab8-46e3-9211-d38e4ae701ce.parquet (1 rows)
Written data to /tmp/part_demo/seg_e47f8bbb-33c2-4795-8924-d5cd5c7d2476.parquet (1 rows)
Written data to /tmp/part_demo/seg_be795f7e-0638-49c7-8f1e-761f3543989d.parquet (1 rows)
Written data to /tmp/part_demo/seg_e29cba0a-c1a6-457d-aef2-3b37c8aadb44.parquet (1 rows)
Written data to /tmp/part_demo/seg_c6f805ad-490d-468c-b2d6-e1ae304ba889.parquet (1 rows)
Written data to /tmp/part_demo/seg_48de31b0-4b2f-4cc5-b32b-a7b6fa64a990.parquet (1 rows)
Written data to /tmp/part_demo/seg_71d85573-5ec8-4c15-a4c9-1ef5b18aa975.parquet (1 rows)
Written data to /tmp/part_demo/seg_5b535e5d-deaa-47e6-89e6-ce3114a43887.parquet (1 rows)
Written data to /tmp/part_demo/seg_68772b63-80fd-4389-8c2a-121d24a17ae2.par

DEBUG: to_manifest_entry: segment_id=seg_7c64db85-6ef7-4370-b82e-c30eb530203e, generated index_files=[IndexFile { file_path: "seg_7c64db85-6ef7-4370-b82e-c30eb530203e.text.inv.parquet", index_type: "inverted", column_name: Some("text"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_7c64db85-6ef7-4370-b82e-c30eb530203e.id.inv.parquet", index_type: "inverted", column_name: Some("id"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_7c64db85-6ef7-4370-b82e-c30eb530203e.category.inv.parquet", index_type: "inverted", column_name: Some("category"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_7c64db85-6ef7-4370-b82e-c30eb530203e.embedding.cluster_0.hnsw.graph", index_type: "vector", column_name: Some("embedding"), blob_type: None, offset: None, length: None }]
DEBUG: Background indexing task: segment=seg_dd9bd59d-2214-45fd-9eac-8d37375b4914.parquet, found index_files=[IndexFile { file_path: "seg_dd9bd59d-22

Committed Manifest: _manifest/v3.json
Committed Manifest: _manifest/v4.json
Committed Manifest: _manifest/v5.json
Committed Manifest: _manifest/v6.json
Committed Manifest: _manifest/v7.json
Committed Manifest: _manifest/v8.json
Committed Manifest: _manifest/v9.json


DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
Successfully attached 4 indexes to manifest for segment seg_33a2a17e-0fa3-4e2a-b9c6-17944755a0ec.parquet
DEBUG: ManifestManager::load_latest: Found version 8 on disk at Path { raw: "_manifest/v8.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: ManifestManager::load_latest: Successfully loaded v7 (entries=0)
DEBUG

...

DEBUG: ManifestManager::load_latest: Found version 95 on disk at Path { raw: "_manifest/v95.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v95 (entries=0)
Successfully attached 4 indexes to manifest for segment seg_5d8839c7-eca5-4e8e-95de-1fa246cb2425.parquet
DEBUG: ManifestManager::load_latest: Found version 96 on disk at Path { raw: "_manifest/v96.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v96 (entries=0)
Successfully attached 4 indexes to manifest for segment seg_467b1929-5143-4441-8c61-281e3a7afd65.parquet
DEBUG: ManifestManager::load_latest: Found version 97 on disk at Path { raw: "_manifest/v97.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v97 (entries=0)
Successfully attached 4 indexes to manifest for segment seg_9bb3955a-8864-4fee-85a2-3ba8964a2c16.parquet
DEBUG: ManifestManager::load_latest: Found version 97 on disk at Path { raw: "_manifest/v97.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v97 (entr

Committed Manifest: _manifest/v97.json
Committed Manifest: _manifest/v98.json
Committed Manifest: _manifest/v99.json


DEBUG: ManifestManager::load_latest: Found version 99 on disk at Path { raw: "_manifest/v99.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v99 (entries=0)
Successfully attached 4 indexes to manifest for segment seg_f8a70501-d89d-40af-a1e4-3fbde3ba749a.parquet
DEBUG: ManifestManager::load_latest: Found version 99 on disk at Path { raw: "_manifest/v99.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v99 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 100 on disk at Path { raw: "_manifest/v100.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v100 (entries=0)
Successfully attached 4 indexes to manifest for segment seg_6d4899fd-81bd-4ffa-be0a-104121f8dfa1.parquet


Committed Manifest: _manifest/v100.json
Committed Manifest: _manifest/v101.json
Committed Manifest: _manifest/v102.json
Partitioned physical layout:


DEBUG: ManifestManager::load_latest: Found version 101 on disk at Path { raw: "_manifest/v101.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v101 (entries=0)
Successfully attached 4 indexes to manifest for segment seg_6efbc46e-fa4d-4fef-8db0-d81a2c3f8617.parquet
DEBUG: ManifestManager::load_latest: Found version 102 on disk at Path { raw: "_manifest/v102.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v102 (entries=0)


## 11. End-to-End RAG Pipeline

Using SQuAD validation data to build a knowledge-backed Q&A system.

In [11]:
print("Loading SQuAD contexts...")
squad = load_dataset("squad", split="validation[0:100]")
squad_df = pd.DataFrame(squad).drop_duplicates(subset=["context"])

squad_emb = model.encode(squad_df["context"].tolist(), show_progress_bar=True)
squad_df["embedding"] = [list(e) for e in squad_emb]

rag_table = hdb.Table("rag_db_presentation")
rag_table.add_index_columns(["embedding", "context"])
rag_table.write(squad_df[["context", "title", "embedding"]])
rag_table.commit()
print("RAG knowledge base initialized.")

Loading SQuAD contexts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Written data to /Users/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.parquet (4 rows)
Building indexes for batch of 4 rows
Indexing String column: context
Indexing Vector column: embedding (type=List(Field { data_type: Float32, nullable: true }))
Indexing Int64 column: __index_level_0__
  Found 4 unique values
Indexing String column: title
  Found 1 unique values
Building HNSW-IVF index (blocking): 4 vectors, 384 dims, use_pq=false
Building HNSW-IVF index: 4 vectors, 1 clusters, 384 dims, M=16, use_pq=false
  [K-Means] Converged early at iteration 2
String Inverted Index written to /Users/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.context.inv.parquet
  - K-Means took: 192.63µs (5 iterations, 1 clusters)
  - Grouping vectors took: 63.92µs
String Inverted Index written to /Users/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.titl

DEBUG: ManifestManager::load_latest: Found version 5 on disk at Path { raw: "_manifest/v5.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v5 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 5 on disk at Path { raw: "_manifest/v5.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v5 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 5 on disk at Path { raw: "_manifest/v5.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v5 (entries=0)
DEBUG: to_manifest_entry: segment_id=seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e, generated index_files=[IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.context.inv.parquet", index_type: "inverted", column_name: Some("context"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.__index_level_0__.inv.parquet", index_type: "inverted", column_name: Some("__index_level_0__"), blob_type: None, offset: None, length: 

## 12. Inference with Retrieval

Connect retrieval to the LLM (requires GROQ_API_KEY).

In [18]:
def ask_hdb(question):
    # Convert question to an embedding (using your local model)
    q_emb = model.encode(question)
    
    # Perform a vector search on your table
    # This is where your table data gets pulled in!
    ctx_df = rag_table.to_pandas(vector_filter={
        "column": "embedding", 
        "query": q_emb, 
        "k": 3  # Get top 3 matching contexts
    })
    
    # Combine retrieved contexts into one string
    contexts = ctx_df["context"].tolist()
    prompt = f"Answer the question based ONLY on this context:\n\n" + "\n".join(contexts) + f"\n\nQuestion: {question}"
    
    # Use a supported Groq model
    model_name = "llama-3.3-70b-versatile"
    
    api_key = os.environ.get("GROQ_TOKEN")
    r = requests.post("https://api.groq.com/openai/v1/chat/completions", 
                      headers={"Authorization": f"Bearer {api_key}"},
                      json={"model": model_name, "messages": [{"role": "user", "content": prompt}]})
    
    res = r.json()
    if "choices" not in res:
        # This will show you the exact error from Groq (e.g., "model_not_found")
        print(f"Groq API Error: {res}")
        return "Error from API"
        
    return res["choices"][0]["message"]["content"]

In [19]:

question = "Where were the first modern Olympic games?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

DEBUG: ManifestManager::load_latest: Found version 8 on disk at Path { raw: "_manifest/v8.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: Total entries to prune: 3
DEBUG: Entry seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.parquet: partitions={}, index_files=[IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.context.inv.parquet", index_type: "inverted", column_name: Some("context"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.__index_level_0__.inv.parquet", index_type: "inverted", column_name: Some("__index_level_0__"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.title.inv.parquet", index_type: "inverted", column_name: Some("title"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.embedding.cluster_0.hnsw.graph", index_type: "vector", column

Q: Where were the first modern Olympic games?
A: The context provided does not mention the location of the first modern Olympic games. The information is about Super Bowl 50, its broadcast, commercial costs, halftime show, and viewership. It does not contain any details about the Olympic games.


In [20]:
question = "What city was superbowl 50 played at?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

DEBUG: ManifestManager::load_latest: Found version 8 on disk at Path { raw: "_manifest/v8.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v8 (entries=0)
DEBUG: Total entries to prune: 3
DEBUG: Entry seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.parquet: partitions={}, index_files=[IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.context.inv.parquet", index_type: "inverted", column_name: Some("context"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.__index_level_0__.inv.parquet", index_type: "inverted", column_name: Some("__index_level_0__"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.title.inv.parquet", index_type: "inverted", column_name: Some("title"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_d2d2d539-d05e-4976-8bcb-3a9c38113a5e.embedding.cluster_0.hnsw.graph", index_type: "vector", column

Q: What city was superbowl 50 played at?
A: The game was played at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California.
